In [29]:
# === [1] ROS + Interface Setup ===
import rospy
import actionlib
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan
from Simulation_pkg.msg import PositionVelocity
import threading
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

# ROS Node
rospy.init_node('jupyter_action_client', anonymous=True, disable_signals=True)

# Globals
position_velocity_pub = rospy.Publisher("position_velocity", PositionVelocity, queue_size=10)
pos_vel_msg = PositionVelocity()
ac = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
ac_lock = threading.Lock()
goal_reached = False
request_coordinates = False
reached_count = 0



In [30]:
# === [2] Interface Widgets ===
status_output = widgets.Output()
position_display = widgets.HTML("<b>Current Position:</b> Waiting for data...")
velocity_display = widgets.HTML("<b>Current Velocity:</b> Waiting for data...")
obstacle_distance_display = widgets.HTML("<b>Closest Obstacle:</b> Waiting for data...")
target_display = widgets.HTML("<b>Target Position:</b> None")

x_input = widgets.FloatText(value=0.0, description='X:')
y_input = widgets.FloatText(value=0.0, description='Y:')

set_goal_button = widgets.Button(description='Set Goal', button_style='success', icon='check')
cancel_button = widgets.Button(description='Cancel Goal', button_style='danger', icon='stop')

In [31]:
# === [3] ROS Callbacks ===
def odom_callback(msg):
    pos_vel_msg.x = msg.pose.pose.position.x
    pos_vel_msg.y = msg.pose.pose.position.y
    pos_vel_msg.vel_x = msg.twist.twist.linear.x
    pos_vel_msg.vel_z = msg.twist.twist.angular.z

    position_display.value = f"<b>Current Position:</b> X: {pos_vel_msg.x:.2f}, Y: {pos_vel_msg.y:.2f}"
    velocity_display.value = f"<b>Current Velocity:</b> Linear: {pos_vel_msg.vel_x:.2f}, Angular: {pos_vel_msg.vel_z:.2f}"
    position_velocity_pub.publish(pos_vel_msg)
    #x_path.append(pos_vel_msg.x)
    #y_path.append(pos_vel_msg.y)

def scan_callback(msg):
    valid_ranges = [r for r in msg.ranges if not rospy.is_shutdown() and not (r == float('inf') or r != r)]
    if valid_ranges:
        closest = min(valid_ranges)
        obstacle_distance_display.value = f"<b>Closest Obstacle:</b> {closest:.2f} meters"
    else:
        obstacle_distance_display.value = "<b>Closest Obstacle:</b> No valid data"

In [32]:
# === [4] Goal Monitor Thread ===
x_path = []
y_path = []

def monitor_goal_status():
    global goal_reached, request_coordinates, reached_count, aborted_count
    rate = rospy.Rate(10)
    while not rospy.is_shutdown():
        with ac_lock:
            if ac:
                state = ac.get_state()
                if state == actionlib.GoalStatus.SUCCEEDED and not goal_reached:
                    reached_count += 1
                    with status_output:
                        clear_output(wait=True)
                        print("Target reached!")
                    goal_reached = True
                    request_coordinates = False
                elif state == actionlib.GoalStatus.ABORTED:
                    with status_output:
                        clear_output(wait=True)
                        print("Target could not be reached. Please set a new goal.")
                    goal_reached = True
                    request_coordinates = False
        rate.sleep()

In [33]:
# === [5] Goal Control Buttons ===
def on_set_goal_button_clicked(b):
    global goal_reached
    x = x_input.value
    y = y_input.value

    if not (isinstance(x, float) and isinstance(y, float) and x == x and y == y):
        with status_output:
            clear_output(wait=True)
            print("Invalid input. Both x and y must be finite numbers.")
        return

    pos_vel_msg.target_x = x
    pos_vel_msg.target_y = y
    goal = PlanningGoal()
    goal.target_pose.header.frame_id = "map"
    goal.target_pose.header.stamp = rospy.Time.now()
    goal.target_pose.pose.position.x = x
    goal.target_pose.pose.position.y = y
    goal.target_pose.pose.orientation.w = 1.0

    with ac_lock:
        ac.send_goal(goal)

    target_display.value = f"<b>Target Position:</b> X: {x:.2f}, Y: {y:.2f}"
    goal_reached = False
    with status_output:
        clear_output(wait=True)
        print(f"Goal sent to ({x:.2f}, {y:.2f}).")

def on_cancel_button_clicked(b):
    global request_coordinates
    with ac_lock:
        ac.cancel_goal()
    with status_output:
        clear_output(wait=True)
        print("Goal canceled.")
    request_coordinates = False

set_goal_button.on_click(on_set_goal_button_clicked)
cancel_button.on_click(on_cancel_button_clicked)

In [34]:
# === [6] Display UI ===
input_widgets = widgets.HBox([x_input, y_input])
button_widgets = widgets.HBox([set_goal_button, cancel_button])
info_widgets = widgets.VBox([
    position_display,
    velocity_display,
    obstacle_distance_display,
    target_display
])
ui = widgets.VBox([info_widgets, input_widgets, button_widgets, status_output])

try:
    print("Waiting for Action server...")
    ac.wait_for_server()
    print("Action server available.")

    monitor_thread = threading.Thread(target=monitor_goal_status)
    monitor_thread.daemon = True
    monitor_thread.start()

    odom_sub = rospy.Subscriber("/odom", Odometry, odom_callback)
    scan_sub = rospy.Subscriber("/scan", LaserScan, scan_callback)

    display(ui)

except rospy.ROSInterruptException:
    print("ROS node interrupted")
except Exception as e:
    print(f"Error: {str(e)}")

Waiting for Action server...
Action server available.


In [38]:
# === [7] Visualization Section Target ===

%matplotlib widget

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display
import ipywidgets as widgets


# Create the plot
fig, ax = plt.subplots()
bar = ax.bar(['Reached'], [reached_count], color='green')
ax.set_title("Reached Goal Count")
ax.set_ylim(0, 10)
ax.set_ylabel("Count")


# Animation update function
def update_bar(frame):
    global reached_count
    bar[0].set_height(reached_count)
    return bar

# ✅ Keep reference to animation to prevent garbage collection
ani = FuncAnimation(fig, update_bar, interval=1000)

# Reset button
reset_button = widgets.Button(description="Reset Reached", button_style='warning')

def reset_reached(b):
    global reached_count
    reached_count = 0

reset_button.on_click(reset_reached)
display(reset_button)






Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Button(button_style='warning', description='Reset Reached', style=ButtonStyle())

In [36]:
%matplotlib widget

import matplotlib.pyplot as plt 
import rospy
import tf
from nav_msgs.msg import Odometry
from tf.transformations import quaternion_matrix 
import numpy as np
from matplotlib.animation import FuncAnimation

class Visualiser:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ln, = plt.plot([], [], 'ro')
        self.x_data, self.y_data = [] , []

    def plot_init(self):
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        return self.ln,

    def odom_callback(self, msg):
        self.y_data.append(msg.pose.pose.position.y)
        self.x_data.append(msg.pose.pose.position.x)

    def update_plot(self, frame):
        self.ln.set_data(self.x_data, self.y_data)
        return self.ln,


# rospy.init_node('odom_visualizer_node') 
vis = Visualiser()
sub = rospy.Subscriber('/odom', Odometry, vis.odom_callback)

ani = FuncAnimation(vis.fig, vis.update_plot, init_func=vis.plot_init) 
plt.show(block=True)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [37]:
# === [8] Cleanup ===
def cleanup():
    try:
        rospy.signal_shutdown("Notebook closing")
        print("ROS node shutdown")
    except:
        pass

import atexit
atexit.register(cleanup)

<function __main__.cleanup()>